# GraphCPP Progress Report #4 — does more MLCPP data help, and for which architecture?

Same environment as the PR2/PR3 notebooks (Kaggle/Colab T4, Python 3.10, torch 1.13.1+cu117, PyG 2.3.0, pytorch-lightning 2.0.1.post0, rdkit 2022.09.5, deepchem 2.7.1).

PR3 asked which **architecture** is best on a fixed dataset. PR4 asks whether the
dataset itself is the limiting factor, by training each architecture twice:

| Arm | Training peptides |
|---|---|
| `original` | only peptides from the original splits |
| `merged` | original + the extra MLCPP peptides, deduplicated |

Both arms are evaluated on **one shared, byte-identical holdout**. That is the whole
design: if each arm had its own test set, a change in score could not be attributed
to the extra data rather than to the change of test set. `run_pr4_comparison.sh`
and cell 9 below both hard-fail if the two holdouts ever differ.

| Experiment | layer_type | mp_layers |
|---|---|---:|
| `graphcpp_sage2` | sageconv | 2 |
| `graphcpp_sage4` (PR2/PR3 reference) | sageconv | 4 |
| `graphcpp_sage6` | sageconv | 6 |
| `graphcpp_transformer2` | transformerconv (4 heads) | 2 |
| `graphcpp_transformer4` | transformerconv (4 heads) | 4 |

Everything else is held fixed: seed 42, hidden_channels=128, mean pooling, topological
torsion fingerprint branch, Adam lr=0.001 + cosine schedule, 100 epochs, best checkpoint
by validation MCC, threshold 0.5.

The merged training set is class-imbalanced (the MLCPP independent set is ~93% negative),
so the merged arm trains with a `pos_weight`-weighted BCE loss read from the dataset
manifest. The `original` arm comes out balanced, so its `pos_weight` is 1.0 and its loss
is identical to PR2/PR3. Validation/test loss is never weighted, so it stays comparable
across arms.

**Before running:** push the branch to GitHub (or change `REPO_BRANCH`), and put the
MLCPP train/test downloads in the repo so `EXTRA_SOURCES` in cell 8 can find them.
Enable a GPU and Internet.

In [ ]:
# 1) Platform, GPU, and Internet check
import os
import sys
import shutil
import subprocess
import urllib.request
from pathlib import Path

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working")
    PLATFORM = "Kaggle"
else:
    WORK_ROOT = Path("/content")
    PLATFORM = "Google Colab"

print("Platform:", PLATFORM)
print("Notebook Python:", sys.version.split()[0])
print("WORK_ROOT:", WORK_ROOT)

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
        text=True,
    )
    print("\nGPU:")
    print(gpu_info)
except Exception as e:
    raise RuntimeError(
        "No NVIDIA GPU is visible. Enable a GPU accelerator, reconnect, and Run All again."
    ) from e

try:
    with urllib.request.urlopen("https://github.com", timeout=15) as r:
        print("Internet access: OK")
except Exception as e:
    raise RuntimeError(
        "Internet access is unavailable. Enable Internet/network access and Run All again."
    ) from e

In [ ]:
# 2) Clean workspace, install Micromamba, and define helpers
ENV_PREFIX = WORK_ROOT / "graphcpp_env"
MAMBA_ROOT = WORK_ROOT / "micromamba_root"
BIN_DIR = WORK_ROOT / "bin"
MM_BIN = BIN_DIR / "micromamba"
PROJECT = WORK_ROOT / "GraphCPP_pr4"

for p in (ENV_PREFIX, MAMBA_ROOT, PROJECT):
    if p.exists():
        shutil.rmtree(p)

if MM_BIN.exists():
    MM_BIN.unlink()

BIN_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    f"curl -fLsS https://micro.mamba.pm/api/micromamba/linux-64/latest "
    f"| tar -xj -C {WORK_ROOT} bin/micromamba",
    shell=True,
    check=True,
)

if not MM_BIN.exists():
    raise RuntimeError("Micromamba installation failed.")

MAMBA_ENV = os.environ.copy()
MAMBA_ENV["MAMBA_ROOT_PREFIX"] = str(MAMBA_ROOT)

def run_checked(cmd, *, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd))
    subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        check=True,
    )

def pip_install(*args):
    run_checked([ENV_PYTHON, "-m", "pip", "install", *args], env=RUNTIME_ENV)

def env_python(*args, cwd=None):
    run_checked([ENV_PYTHON, *args], cwd=cwd, env=RUNTIME_ENV)

run_checked([MM_BIN, "--version"], env=MAMBA_ENV)

In [ ]:
# 3) Create isolated Python 3.10 environment
run_checked(
    [
        MM_BIN, "create", "-y",
        "-p", ENV_PREFIX,
        "-c", "conda-forge",
        "python=3.10",
        "pip",
    ],
    env=MAMBA_ENV,
)

ENV_PYTHON = str(ENV_PREFIX / "bin" / "python")
if not Path(ENV_PYTHON).exists():
    raise RuntimeError(f"Environment Python missing: {ENV_PYTHON}")

RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV["PATH"] = f"{ENV_PREFIX / 'bin'}:{RUNTIME_ENV['PATH']}"
RUNTIME_ENV["PYTHONNOUSERSITE"] = "1"

pip_install(
    "--upgrade",
    "pip==24.0",
    "setuptools==69.5.1",
    "wheel==0.43.0",
)

env_python("--version")

In [ ]:
# 4) PyTorch 1.13.1 + CUDA 11.7
pip_install(
    "torch==1.13.1+cu117",
    "torchvision==0.14.1+cu117",
    "torchaudio==0.13.1",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cu117",
)

In [ ]:
# 5) Scientific / chemistry stack
# Use Micromamba for the old scientific stack to avoid pip resolver problems.
run_checked(
    [
        MM_BIN, "install", "-y",
        "-p", ENV_PREFIX,
        "-c", "conda-forge",
        "numpy=1.24.4",
        "pandas=1.5.3",
        "scipy=1.10.1",
        "scikit-learn=1.2.2",
        "joblib=1.2.0",
        "sympy=1.11.1",
        "tqdm=4.65.0",
        "rdkit=2022.09.5",
        "python-dateutil=2.8.2",
        "pytz=2023.3",
    ],
    env=MAMBA_ENV,
)

# GraphCPP uses DeepChem's graph featurizer.
pip_install("--no-deps", "deepchem==2.7.1")

In [ ]:
# 6) PyTorch Lightning + PyTorch Geometric
pip_install(
    "pytorch-lightning==2.0.1.post0",
    "torchmetrics==0.11.4",
    "lightning-utilities==0.8.0",
)

pip_install("torch-geometric==2.3.0")

pip_install(
    "torch-scatter==2.1.1",
    "torch-sparse==0.6.17",
    "-f",
    "https://data.pyg.org/whl/torch-1.13.1+cu117.html",
)

In [ ]:
# 7) Clone the branch
REPO_URL = "https://github.com/finmook/GraphCPP.git"
REPO_BRANCH = "pr4-merged-data"

run_checked(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, PROJECT])
env_python("-m", "compileall", "-q", str(PROJECT), cwd=PROJECT)
print("PROJECT:", PROJECT)

## Build the merged dataset

`prepare_merged_dataset.py` pools every source, deduplicates by **canonical SMILES**
(not by name — the naming schemes differ between files), refuses to continue if the
same peptide appears with conflicting labels, carves one balanced holdout, and writes
both arms against it.

List the MLCPP downloads in `EXTRA_SOURCES`. A CSV needs a `label` column plus either
`smiles` or `sequence`; FASTA labels are read from the headers
(`positive`/`negative`/`CPP`/`non-CPP`), or append `:0` / `:1` to force one label for a
whole file.

Peptides whose amino-acid sequence cannot be recovered from SMILES (cyclic or modified)
are routed to training only, because `evaluate_pr3.py` needs `sequence`/`len` on the
test split for the length-bin analysis.

In [ ]:
# 8) Merge and deduplicate. Dry run first -- nothing is written.
EXTRA_SOURCES = [
    "dataset/mlcpp2_independent.csv",
    # Add the MLCPP train/test downloads here, for example:
    # "dataset/mlcpp_raw/mlcpp_train.fasta",
    # "dataset/mlcpp_raw/mlcpp_independent_positive.fasta:1",
]
DATA_ROOT = PROJECT / "dataset" / "raw" / "pr4"

extra_args = []
for src in EXTRA_SOURCES:
    # Strip any trailing :0 / :1 label override before checking the file exists.
    path = PROJECT / src.rsplit(":", 1)[0]
    if not path.is_file():
        raise FileNotFoundError(f"{path} is missing -- add it to the repo or drop it from EXTRA_SOURCES")
    extra_args += ["--extra", src]

env_python("prepare_merged_dataset.py", *extra_args, "--out", str(DATA_ROOT), "--dry-run", cwd=PROJECT)


In [ ]:
# 9) Counts look right? Write both arms, then verify the shared holdout.
env_python("prepare_merged_dataset.py", *extra_args, "--out", str(DATA_ROOT), cwd=PROJECT)

import filecmp
import json

ARMS = ["original", "merged"]

# The premise of the entire comparison. Check it, do not assume it.
for split in ["val", "test"]:
    identical = filecmp.cmp(DATA_ROOT / "original" / f"{split}.csv",
                            DATA_ROOT / "merged" / f"{split}.csv", shallow=False)
    print(f"{split}.csv identical across arms: {identical}")
    if not identical:
        raise RuntimeError(f"{split}.csv differs between arms -- the comparison would be confounded")

POS_WEIGHT = {}
for arm in ARMS:
    manifest = json.loads((DATA_ROOT / arm / "manifest.json").read_text())
    POS_WEIGHT[arm] = manifest["pos_weight"]
    print(f"\n{arm}: pos_weight={manifest['pos_weight']:.4f}")
    for split, counts in manifest["arms"][arm].items():
        print(f"  {split:>5}: {counts['rows']:>5} rows  "
              f"{counts['positive']} pos / {counts['negative']} neg")
print("\nDuplicates removed:", manifest["duplicates_removed"])
print("Unique peptides:", manifest["unique_peptides"])

In [ ]:
# 10) Sanity check: one real batch through every architecture (also processes the test split)
env_python("-c", "import torch; print('CUDA available:', torch.cuda.is_available()); assert torch.cuda.is_available()", cwd=PROJECT)
env_python("sanity_check.py", "--dataset", str(DATA_ROOT / "merged"), cwd=PROJECT)

## Train: 5 architectures x 2 arms = 10 runs

Featurization of each split is cached in `<arm>/processed/` on first use, so the first
run of each arm is slower than the rest.

In [ ]:
# 11) Train
EXPERIMENTS = [
    "graphcpp_sage2",
    "graphcpp_sage4",
    "graphcpp_sage6",
    "graphcpp_transformer2",
    "graphcpp_transformer4",
]

for arm in ARMS:
    for name in EXPERIMENTS:
        print("\n" + "=" * 100 + f"\nTRAINING: {name}  (arm={arm}, pos_weight={POS_WEIGHT[arm]:.4f})\n" + "=" * 100)
        log_path = PROJECT / f"{arm}_{name}_run.log"
        with log_path.open("w", encoding="utf-8") as log_file:
            proc = subprocess.Popen(
                [ENV_PYTHON, "main.py",
                 "--seed", "42",
                 "--config", name,
                 "--dataset", str(DATA_ROOT / arm),
                 "--results_dir", f"results/pr4_{arm}",
                 "--pos_weight", str(POS_WEIGHT[arm])],
                cwd=str(PROJECT), env=RUNTIME_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
            )
            for line in proc.stdout:
                if "it/s" in line or "s/it" in line:
                    continue
                log_file.write(line)
                if line.startswith(("Seed", "Experiment", "layer_type", "Parameter count", "Test MCC")) or "test_" in line:
                    print(line, end="")
            if proc.wait() != 0:
                raise RuntimeError(f"{arm}/{name} failed. See {log_path}")

## Evaluation

Each arm gets the usual PR3 tables (overall, length bins, paired-vs-reference), and
then the two arms are differenced per architecture. Because the holdout is shared,
those deltas are the actual effect of the extra training data.

In [ ]:
# 12) Per-arm tables
import pandas as pd
from IPython.display import display

for arm in ARMS:
    print("\n" + "#" * 100 + f"\n# ARM: {arm}\n" + "#" * 100)
    env_python("evaluate_pr3.py", "--results_dir", f"results/pr4_{arm}", "--models", *EXPERIMENTS, cwd=PROJECT)
    display(pd.read_csv(PROJECT / "results" / f"pr4_{arm}" / "pr3_overall_comparison.csv"))

In [ ]:
# 13) The headline: MCC change per architecture, on the shared holdout
frames = []
for arm in ARMS:
    df = pd.read_csv(PROJECT / "results" / f"pr4_{arm}" / "pr3_overall_comparison.csv")
    frames.append(df.assign(arm=arm))
both = pd.concat(frames)

summary = both.pivot(index="Model", columns="arm", values=["MCC", "Accuracy", "ROC-AUC"])
delta = pd.DataFrame({
    "MCC original": summary[("MCC", "original")],
    "MCC merged": summary[("MCC", "merged")],
    "MCC delta": summary[("MCC", "merged")] - summary[("MCC", "original")],
    "AUC original": summary[("ROC-AUC", "original")],
    "AUC merged": summary[("ROC-AUC", "merged")],
    "AUC delta": summary[("ROC-AUC", "merged")] - summary[("ROC-AUC", "original")],
}).sort_values("MCC delta", ascending=False)

pd.set_option("display.width", 250)
print("Effect of the extra MLCPP training data (shared holdout, threshold 0.5)\n")
display(delta.style.format("{:.4f}"))
delta.to_csv(PROJECT / "results" / "pr4_data_effect.csv")

In [ ]:
# 14) Copy results (+ run logs) out of the project and zip them
OUT = WORK_ROOT / "pr4_results"
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

for arm in ARMS:
    shutil.copytree(PROJECT / "results" / f"pr4_{arm}", OUT / f"pr4_{arm}")
    shutil.copy(DATA_ROOT / arm / "manifest.json", OUT / f"manifest_{arm}.json")
shutil.copy(PROJECT / "results" / "pr4_data_effect.csv", OUT / "pr4_data_effect.csv")
shutil.copytree(PROJECT / "logs", OUT / "logs")
for log in PROJECT.glob("*_run.log"):
    shutil.copy(log, OUT / log.name)

shutil.make_archive(str(WORK_ROOT / "pr4_results"), "zip", root_dir=OUT)
print("Saved:", OUT, "and", WORK_ROOT / "pr4_results.zip")

In [ ]:
# 15) Remove the large environment/repo copies so the saved notebook Output only contains the results.
# (Run this last. Re-running training afterwards requires Run All again.)
for p in (ENV_PREFIX, MAMBA_ROOT, BIN_DIR, PROJECT):
    if p.exists():
        shutil.rmtree(p)
for p in WORK_ROOT.iterdir():
    print(p)

## How to interpret

- **The deltas in cell 13 are the result.** Both arms saw the same holdout peptides, so
  a positive `MCC delta` means the extra data helped that architecture.
- **Check whether the deltas differ across architectures.** If every architecture gains
  about the same amount, the data helped generically. If the deeper or attention-based
  models gain more, that is evidence they were data-limited in PR2/PR3 — which is the
  interesting claim.
- **The MLCPP independent set is ~93% negative**, so most of what the merged arm gains is
  negative examples. Expect precision/specificity to improve more than recall. A large
  accuracy gain with a flat MCC would just be the classifier leaning negative.
- **`graphcpp_sage4` in the `original` arm is not directly comparable to the PR2 number
  (0.6389)** — the holdout changed, since re-splitting was needed to keep the two arms
  honest. Compare within this notebook, not against PR2/PR3 absolute values.
- **Both arms re-trained in one session.** GPU nondeterminism still causes small run-to-run
  differences, so treat a delta of a few thousandths of MCC as noise, not signal.